In [ ]:

import pandas as pd
import sqlite3
import logging
import time
from pathlib import Path
from datetime import datetime
from IPython.display import display

print("All required libraries imported successfully!")


All required libraries imported successfully!


In [ ]:

# Configure logging
logging.basicConfig(
    filename="etl_pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

# Configure file paths
CSV_FILE = "cleaned_dataset-2.csv"
DATABASE_FILE = "datheon_database.db"

logging.info("ETL pipeline configuration initialized.")

print("Logging configured successfully!")
print("CSV file:", CSV_FILE)
print("Database file:", DATABASE_FILE)


Logging configured successfully!
CSV file: cleaned_dataset-2.csv
Database file: datheon_database.db


In [ ]:

import os

print("Current files:")
print(os.listdir("/content"))

if os.path.exists(CSV_FILE):
    print("\nCSV file found!")
    df = pd.read_csv(CSV_FILE)
    print("Dataset shape:", df.shape)
    display(df.head())
else:
    print("\nCSV file not found. Upload your cleaned CSV to Colab.")


Current files:
['.config', 'cleaned_dataset-2.csv', 'etl_pipeline.log', 'sample_data']

CSV file found!
Dataset shape: (7, 6)


,product_id,product_name,price,date,category,currency
0,101,Laptop,50000.0,2026-01-10,Electronics,INR
1,102,Phone,25000.0,2026-02-11,Electronics,Unspecified
2,103,Tablet,18500.0,2026-03-15,Electronics,INR
3,104,Monitor,NaN,2026-04-15,Display,Unspecified
4,105,Keyboard,100.0,2026-05-05,Accessories,USD


In [ ]:

# Extract the dataset
df = pd.read_csv(CSV_FILE)

print("Dataset loaded successfully!")
print("Total rows:", len(df))
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())


Dataset loaded successfully!
Total rows: 7

Column names:
['product_id', 'product_name', 'price', 'date', 'category', 'currency']

First 5 records:


,product_id,product_name,price,date,category,currency
0,101,Laptop,50000.0,2026-01-10,Electronics,INR
1,102,Phone,25000.0,2026-02-11,Electronics,Unspecified
2,103,Tablet,18500.0,2026-03-15,Electronics,INR
3,104,Monitor,NaN,2026-04-15,Display,Unspecified
4,105,Keyboard,100.0,2026-05-05,Accessories,USD


In [ ]:

import os

print(os.listdir("/content"))


['.config', 'cleaned_dataset-2.csv', 'etl_pipeline.log', 'sample_data']


In [ ]:

import os

print("CSV found:", os.path.exists("cleaned_dataset-2.csv"))


CSV found: True


In [ ]:

def run_etl():
    try:
        logging.info("ETL pipeline started")

        # 1. Extract
        data = pd.read_csv(CSV_FILE)
        logging.info("Extracted %s rows", len(data))

        # 2. Transform
        data.columns = data.columns.str.strip().str.lower()

        required = [
            "product_id", "product_name", "price",
            "date", "category", "currency"
        ]

        missing = set(required) - set(data.columns)
        if missing:
            raise ValueError(f"Missing columns: {missing}")

        data = data.drop_duplicates(subset=["product_id"])
        data["product_name"] = data["product_name"].astype("string").str.strip()
        data["category"] = data["category"].fillna("Unknown").astype(str).str.strip()
        data["currency"] = data["currency"].replace(
            "Unspecified", pd.NA
        ).fillna("INR")

        data["price"] = pd.to_numeric(data["price"], errors="coerce")
        data["date"] = pd.to_datetime(
            data["date"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")

        # Remove records missing essential fields
        data = data.dropna(
            subset=["product_id", "product_name", "price", "date"]
        )

        if (data["price"] < 0).any():
            raise ValueError("Negative prices found")

        # 3. Load using an SQLite upsert
        with sqlite3.connect(DATABASE_FILE) as conn:
            data.to_sql(
                "products",
                conn,
                if_exists="replace",
                index=False
            )

        logging.info("Loaded %s rows", len(data))
        print(f"ETL completed successfully! Rows loaded: {len(data)}")

    except Exception as error:
        logging.exception("ETL pipeline failed")
        print("ETL failed:", error)
        raise


In [ ]:
run_etl()

ETL completed successfully! Rows loaded: 5


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("datheon_database.db")

df_check = pd.read_sql_query("SELECT * FROM products", conn)

print("Total rows in database:", len(df_check))
display(df_check)

conn.close()

Total rows in database: 5


,product_id,product_name,price,date,category,currency
0,101,Laptop,50000.0,2026-01-10,Electronics,INR
1,102,Phone,25000.0,2026-02-11,Electronics,INR
2,103,Tablet,18500.0,2026-03-15,Electronics,INR
3,105,Keyboard,100.0,2026-05-05,Accessories,USD
4,107,Mouse,1500.0,2026-06-20,Uncategorized,INR


In [ ]:
import schedule
import time
import logging

def scheduled_etl():
    try:
        print("Scheduled ETL started...")
        run_etl()
        print("Scheduled ETL finished!")
    except Exception as e:
        logging.exception("Scheduled ETL failed")
        print("Error:", e)

# Run ETL every day at 9:00 AM
schedule.every().day.at("09:00").do(scheduled_etl)

print("Scheduler started. ETL will run daily at 09:00.")

Scheduler started. ETL will run daily at 09:00.


In [ ]:
!pip install schedule

In [ ]:
import logging
from datetime import datetime
import os

LOG_FILE = "etl_pipeline.log"

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

def run_scheduled_etl():
    start_time = datetime.now()

    try:
        logging.info("ETL pipeline started")

        run_etl()

        duration = (datetime.now() - start_time).total_seconds()
        logging.info(
            "ETL pipeline completed successfully in %.2f seconds",
            duration
        )

        print("ETL completed successfully!")
        print(f"Execution time: {duration:.2f} seconds")

    except Exception:
        logging.exception("ETL pipeline failed")
        print("ETL failed. Check the log file.")

run_scheduled_etl()

ETL completed successfully! Rows loaded: 5
ETL completed successfully!
Execution time: 0.08 seconds


In [ ]:

from datetime import datetime
import logging

def run_etl_with_alert():
    start_time = datetime.now()

    try:
        logging.info("ETL pipeline started")
        run_etl()

        duration = (datetime.now() - start_time).total_seconds()

        message = (
            "ETL SUCCESS\n"
            f"Time: {datetime.now()}\n"
            f"Duration: {duration:.2f} seconds"
        )

        print(message)
        logging.info(message)

    except Exception as error:
        message = f"ETL FAILED: {error}"
        print(message)
        logging.exception(message)

run_etl_with_alert()


ETL completed successfully! Rows loaded: 5
ETL SUCCESS
Time: 2026-10-25 16:04:16.983156
Duration: 0.04 seconds


In [26]:

with open("etl_pipeline.log", "r") as log_file:
    print(log_file.read()[-3000:])


2026-10-25 15:59:34,050 - INFO - ETL pipeline configuration initialized.
2026-10-25 16:00:03,133 - INFO - ETL pipeline started
2026-10-25 16:00:03,137 - INFO - Extracted 7 rows
2026-10-25 16:00:03,170 - INFO - Loaded 5 rows
2026-10-25 16:04:06,141 - INFO - ETL pipeline started
2026-10-25 16:04:06,141 - INFO - ETL pipeline started
2026-10-25 16:04:06,149 - INFO - Extracted 7 rows
2026-10-25 16:04:06,219 - INFO - Loaded 5 rows
2026-10-25 16:04:06,220 - INFO - ETL pipeline completed successfully in 0.08 seconds
2026-10-25 16:04:16,938 - INFO - ETL pipeline started
2026-10-25 16:04:16,938 - INFO - ETL pipeline started
2026-10-25 16:04:16,943 - INFO - Extracted 7 rows
2026-10-25 16:04:16,982 - INFO - Loaded 5 rows
2026-10-25 16:04:16,983 - INFO - ETL SUCCESS
Time: 2026-10-25 16:04:16.983156
Duration: 0.04 seconds



In [27]:

import os

files_to_check = [
    CSV_FILE,
    DATABASE_FILE,
    "etl_pipeline.log"
]

print("FINAL PROJECT FILE CHECK")
print("-" * 30)

for file in files_to_check:
    if os.path.exists(file):
        print(f"FOUND: {file}")
    else:
        print(f"MISSING: {file}")


FINAL PROJECT FILE CHECK
------------------------------
FOUND: cleaned_dataset-2.csv
FOUND: datheon_database.db
FOUND: etl_pipeline.log


In [28]:

from google.colab import files

files.download("cleaned_dataset-2.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>